# LAB GenAI — LLMs — OpenAI GPT API Exercises

Completed exercises with brief parameter explanations and an optional Google Vertex AI bonus. **Run setup first**, then run individual API cells. API calls may incur charges. Responses vary and have not been executed in this supplied notebook.

**API compatibility:** The exercises mix modern chat parameters with legacy text-completions parameters (`best_of`, `echo`). They are documented below but not passed to unsupported endpoints. Parameters are also model-dependent.

References: [OpenAI Chat API](https://developers.openai.com/api/reference/resources/chat), [OpenAI legacy Completions](https://developers.openai.com/api/reference/resources/completions/methods/create), [Google Gen AI SDK](https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/start/libraries), [Vertex generation parameters](https://docs.cloud.google.com/gemini-enterprise-agent-platform/reference/models/inference).

## Setup
Install dependencies if needed, then set `OPENAI_API_KEY` in your local environment (do not paste keys into notebooks or commit them to Git). We use `gpt-4o-mini` for the sampling-parameter exercises; confirm model availability in your account.

In [ ]:
# Run once if needed:
# %pip install -U openai google-genai

In [ ]:
import os
from openai import OpenAI

# Example: set OPENAI_API_KEY using your system environment, not source code.
# In PowerShell (before launching Jupyter): $env:OPENAI_API_KEY="..."
if not os.getenv("OPENAI_API_KEY"):
    print("Set OPENAI_API_KEY before running the OpenAI API cells.")
    client = None
else:
    client = OpenAI()

OPENAI_MODEL = "gpt-4o-mini"

def chat(prompt, system="You are a helpful assistant.", **params):
    """Make one OpenAI chat request, returning the response text."""
    if client is None:
        raise RuntimeError("Please set OPENAI_API_KEY and rerun the setup cell.")
    result = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[{"role": "system", "content": system},
                  {"role": "user", "content": prompt}],
        **params
    )
    return result.choices[0].message.content or ""

## OpenAI parameter reference

| Parameter | What changes when you adjust it? |
|---|---|
| `temperature` | Higher tends to make output more varied; lower tends to make it more focused. |
| `top_p` | Nucleus sampling: lower values restrict the possible token choices. Usually tune **either** `temperature` **or** `top_p`. |
| `max_tokens` | Upper bound on generated tokens (for this chat-compatible model). Other models may use `max_completion_tokens`. |
| `frequency_penalty` | Positive values discourage repeating tokens frequently. |
| `presence_penalty` | Positive values encourage introducing topics/tokens not yet used. |
| `n` | Number of alternative answers generated; costs can increase. |
| `stop` | Ends generation when the specified text appears; not supported by every model. |
| `logprobs` | Returns token-level log probabilities on compatible chat models; it is metadata, not a creativity setting. |
| `logit_bias` | Alters probability of **specific token IDs**; not ordinary words. Requires a compatible model and tokenizer. |
| `best_of` | Legacy Completions parameter (server generates more candidates and chooses one); **not** supported by Chat Completions. |
| `echo` | Legacy Completions option to return the input prompt with the output; **not** supported by Chat Completions. You can display the prompt yourself. |

**Note:** `n`, `stop`, token limits and probability features have model-specific restrictions. The examples keep requests small; parameter changes do not guarantee any exact result.

## 1. Basic Conversation
**Exercise:** Build a chatbot on a topic and experiment with `temperature`, `max_tokens`, `top_p`, `frequency_penalty`, `presence_penalty`, `n`, `stop`.

In [ ]:
topic = "the history of computers"
question = "What was the importance of the first programmable computers?"
answer = chat(
    question, system=f"You are a friendly tutor specializing in {topic}.",
    temperature=0.3, max_tokens=180, top_p=1,
    frequency_penalty=0, presence_penalty=0, n=1,
    stop=["END_OF_ANSWER"]
)
print(answer)

In [ ]:
# Compare the response style at different temperatures.
for t in [0.0, 0.7, 1.2]:
    print(f"\n--- temperature={t} ---")
    print(chat("Explain what an algorithm is in two sentences.",
               temperature=t, max_tokens=90))

**Parameter observations (expected, not measured here):** At `temperature=0` responses are usually more consistent; at `1.2` they can be more diverse. Lower `max_tokens` may truncate the answer. Lower `top_p` restricts sampling. Positive frequency penalties can reduce repetitive wording; positive presence penalties favor new material. `n=3` asks for three choices (the simple `chat()` helper displays only the first); `stop` truncates when an exact stop sequence is generated. Avoid changing temperature and top-p simultaneously when isolating effects.

## 2. Summarization
**Exercise:** Summarize a long input into a few sentences. Explore `temperature`, `max_tokens`, `top_p`, `frequency_penalty`, `presence_penalty`, `best_of`, `logprobs`.

In [ ]:
long_text = """
Machine learning is a field of artificial intelligence that builds systems capable
of learning patterns from data. Supervised learning uses labeled examples to train
models for classification and regression. Unsupervised learning seeks structure
in unlabeled data, for example through clustering. Model evaluation is essential:
a model that performs very well on training data can still fail on new observations.
Using held-out test data helps estimate how well a model generalizes.
"""
summary = chat("Summarize the following text in 2-3 sentences:\n" + long_text,
               temperature=0.2, max_tokens=120, top_p=1,
               frequency_penalty=0, presence_penalty=0)
print(summary)

In [ ]:
# Examine log probabilities if supported by the chosen model.
# This is optional metadata; it does not directly make the summary better.
if client is None:
    print("Configure API key first.")
else:
    response = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[{"role": "user", "content": "Summarize in one sentence: " + long_text}],
        temperature=0.2, max_tokens=75, logprobs=True, top_logprobs=2
    )
    print(response.choices[0].message.content)
    print("Log probabilities available:", response.choices[0].logprobs is not None)

**Parameter observations:** Lower `temperature` usually gives more predictable summaries; a short `max_tokens` may omit important ideas. Lower `top_p` reduces token variety. Positive penalties can discourage repeated words but may distort faithful summaries. `logprobs=True` exposes token probability information if supported. **`best_of` is only a legacy Completions parameter**, not available in this modern chat request; comparing multiple manually generated summaries is an alternative.

## 3. Translation
**Exercise:** Translate between languages. Explore `temperature`, `max_tokens`, `top_p`, `frequency_penalty`, `presence_penalty`, `echo`, `logit_bias`.

In [ ]:
source = "Artificial intelligence can support scientists in analyzing large datasets."
target_language = "German"
translated = chat(
    f"Translate this English text into {target_language}. Output only the translation:\n{source}",
    temperature=0, max_tokens=90, top_p=1,
    frequency_penalty=0, presence_penalty=0
)
print("Original:", source)  # Demonstrates the effect of legacy `echo` client-side.
print("Translation:", translated)

**Parameter observations:** Low temperature and a generous enough token limit usually suit faithful translation. Increasing temperature/top-p may increase stylistic variation, but accuracy can suffer. Penalties may discourage needed repetitions (e.g. names). `echo` belongs to legacy Completions; here we simply print the source explicitly. `logit_bias` controls individual **token IDs** and can force or discourage wording, but it is tokenizer-specific and can harm translation; it is not necessary for this exercise.

## 4. Sentiment Analysis
**Exercise:** Classify a sentence as positive, negative or neutral. Explore `temperature`, `max_tokens`, `top_p`, `frequency_penalty`, `presence_penalty`, `n`, `logprobs`.

In [ ]:
reviews = [
    "I loved the service and would definitely return!",
    "The product broke after one day. I'm very disappointed.",
    "The parcel arrived on Tuesday."
]
for review in reviews:
    label = chat(
        f"Classify the sentiment as positive, negative, or neutral. "
        f"Reply with exactly one label. Text: {review}",
        temperature=0, max_tokens=10, top_p=1,
        frequency_penalty=0, presence_penalty=0, n=1
    )
    print(f"{review} -> {label.strip()}")

**Parameter observations:** Low temperature helps make sentiment labels more consistent, but it does not guarantee correctness. A very small token limit may truncate the label. Sampling (`top_p`) and penalties generally matter less for a one-word classification. `n>1` produces alternative classifications that might disagree. `logprobs` can provide token-level scores on supported models; these are not automatically calibrated sentiment confidence probabilities.

## 5. Text Completion
**Exercise:** Continue a text prompt. Explore `temperature`, `max_tokens`, `top_p`, `frequency_penalty`, `presence_penalty`, `stop`, `best_of`.

In [ ]:
starter = "In a city where all traffic lights were controlled by AI, one morning"
continuation = chat(
    "Continue this story in one creative paragraph. Begin directly with the continuation:\n" + starter,
    system="You write original short fiction.",
    temperature=0.9, max_tokens=160, top_p=0.95,
    frequency_penalty=0.3, presence_penalty=0.2,
    stop=["THE END"]
)
print(starter, continuation)

**Parameter observations:** Higher temperature or top-p can make continuations more surprising. `max_tokens` limits length, a positive frequency penalty can reduce repetition, and a positive presence penalty can encourage novel content. `stop` can end a response before its token limit. `best_of` was a legacy Completions feature; for Chat Completions, you can request `n` alternatives and select one yourself.

---
# BONUS — Google Vertex AI

Uses Google's **Gen AI Python SDK** instead of the retired `vertexai.generative_models` interface. You need a Google Cloud project with Vertex AI enabled, billing/permissions configured, and Application Default Credentials (ADC). Set `GOOGLE_CLOUD_PROJECT` and optionally `GOOGLE_CLOUD_LOCATION` in your environment. The bonus cells call an API only when you run them.

**Differences:** `max_output_tokens` corresponds broadly to the output-token cap; `candidate_count` is the closest equivalent to `n` but supported candidate counts vary by model. `stop_sequences` corresponds to `stop`. `best_of`, `echo`, and OpenAI-specific `logit_bias` are not Vertex Gemini generation parameters. `logprobs`/`response_logprobs` are model-specific and are being deprecated for some newer Gemini model families.

In [ ]:
# Run if necessary: %pip install -U google-genai
from google import genai
from google.genai import types

PROJECT = os.getenv("GOOGLE_CLOUD_PROJECT")
LOCATION = os.getenv("GOOGLE_CLOUD_LOCATION", "us-central1")
VERTEX_MODEL = "gemini-2.5-flash"
vertex_client = genai.Client(vertexai=True, project=PROJECT, location=LOCATION) if PROJECT else None

def vertex_chat(prompt, **settings):
    if vertex_client is None:
        raise RuntimeError("Set GOOGLE_CLOUD_PROJECT and configure Google Cloud ADC first.")
    response = vertex_client.models.generate_content(
        model=VERTEX_MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(**settings)
    )
    return response.text or ""

## Bonus 1. Basic Conversation

In [ ]:
print(vertex_chat("In two sentences, explain how neural networks learn.",
                  temperature=0.4, max_output_tokens=140, top_p=0.95,
                  frequency_penalty=0, presence_penalty=0,
                  stop_sequences=["END_OF_ANSWER"]))

**What changes:** Higher temperature/top-p usually increases answer variation; lower output tokens can shorten or truncate answers. Positive penalties discourage reused words/topics where supported. Vertex uses `candidate_count` for multiple responses rather than `n`, but some model combinations reject values greater than 1. `stop_sequences` provides text-based cutoffs.

## Bonus 2. Summarization

In [ ]:
print(vertex_chat("Summarize in two sentences:\n" + long_text,
                  temperature=0.2, max_output_tokens=140, top_p=0.95))

**What changes:** Low temperature and a suitable output limit favor stable summaries. Repetition penalties can hurt fidelity. Vertex does not offer a direct `best_of` equivalent. `response_logprobs` and `logprobs` may be offered for selected Gemini models, but their support and deprecation status vary, so this notebook leaves them out of the runnable calls.

## Bonus 3. Translation

In [ ]:
print(vertex_chat("Translate the following to French. Return only the translation: " + source,
                  temperature=0, max_output_tokens=100, top_p=1))

**What changes:** Low temperature generally supports more consistent translations. A low token cap can truncate a long sentence. `echo` is not a Vertex Gemini option; print source text separately if needed. `logit_bias` is not a Vertex Gemini generation option.

## Bonus 4. Sentiment Analysis

In [ ]:
for review in reviews:
    result = vertex_chat("Reply with one of positive, negative, neutral: " + review,
                         temperature=0, max_output_tokens=20)
    print(review, "->", result.strip())

**What changes:** Low temperature makes label output typically more stable. `candidate_count` (where supported) can request alternatives instead of `n`. `logprobs` is diagnostic metadata, not sentiment accuracy, and availability varies by model. Frequency and presence penalties are of limited use in one-word labels.

## Bonus 5. Text Completion

In [ ]:
print(vertex_chat("Continue this story in a short creative paragraph: " + starter,
                  temperature=0.9, top_p=0.95, max_output_tokens=200,
                  frequency_penalty=0.2, presence_penalty=0.2,
                  stop_sequences=["THE END"]))

**What changes:** Higher temperature/top-p may help creativity. Repetition penalties can change wording, and stop sequences can cut off text early. `best_of` is unavailable; manual candidate comparison is possible. When changing parameters for a fair comparison, keep the prompt and model fixed and change one setting at a time.

## Conclusion
Temperature, sampling and penalties affect **how** text is generated; token caps affect **how much** is generated. Classification and translation often benefit from lower randomness, while creative text may benefit from more. Always verify the current supported parameters for the specific model, because APIs evolve. No API results are pre-filled or fabricated in this notebook.